## Objective

Construct the final integrated modeling cohort for downstream multi-omic and pharmacogenomic analyses.

The previous notebooks established the structurally harmonized DepMap–GDSC model universe. This notebook evaluates whether those harmonized models have sufficient data availability across the molecular and pharmacological layers required for downstream analysis.

Specifically, this notebook assesses:

- expression data availability;
- somatic mutation data availability;
- DepMap omics profile availability;
- GDSC pharmacological response availability;
- overlap between molecular and pharmacological data layers;
- model retention after applying explicit inclusion criteria.

The main output is a reproducible definition of the integrated modeling cohort.

## Scope

This notebook is part of the infrastructure layer of the project.

It does not perform biological interpretation, dimensionality reduction, clustering, drug selection, resistance-like phenotype definition, signature construction, or predictive modeling.

Those analyses will be performed only after the usable model universe has been explicitly defined.

## Conceptual distinction

The harmonized model universe answers:

> Which DepMap and GDSC models can be structurally matched?

The integrated modeling cohort answers:

> Which harmonized models are actually usable for downstream multi-omic and pharmacogenomic analysis?

This distinction is important because structural overlap does not imply analytical usability.

A model may be harmonized across resources but still lack expression data, mutation data, omics profile metadata, or sufficient pharmacological response information.

## Expected outputs

This notebook will generate:

```text
data/interim/22_integrated_modeling_cohort.csv
data/interim/22_cohort_summary.csv
data/interim/22_model_layer_coverage.csv
```

## Methodological note

All inclusion criteria are defined explicitly and applied after assessing data-layer coverage.

The final cohort should be interpreted as a computationally defined analysis universe, not as a clinically validated cohort or a causal model of drug resistance.

---

In [1]:
# =============================================================================
# Imports
# =============================================================================

from pathlib import Path

import pandas as pd
import numpy as np

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Load input datasets

Load the harmonized model universe together with the molecular and
pharmacological resources required for coverage assessment.

In [2]:
# =============================================================================
# Paths
# =============================================================================

PROJECT_ROOT = Path.cwd().parents[1]

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

DEPMAP_DIR = PROJECT_ROOT / "data" / "raw" / "depmap"
GDSC_DIR = PROJECT_ROOT / "data" / "raw" / "gdsc"

# =============================================================================
# Input files
# =============================================================================

harmonized_path = (
    INTERIM_DIR /
    "21_harmonized_model_universe.csv"
)

expression_path = (
    DEPMAP_DIR /
    "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"
)

mutations_path = (
    DEPMAP_DIR /
    "OmicsSomaticMutations.csv"
)

omics_profiles_path = (
    DEPMAP_DIR /
    "OmicsProfiles.csv"
)

gdsc_path = (
    GDSC_DIR /
    "GDSC2_fitted_dose_response_27Oct23.xlsx"
)

# =============================================================================
# Load datasets
# =============================================================================

harmonized_models = pd.read_csv(harmonized_path)

expression = pd.read_csv(expression_path)

mutations = pd.read_csv(mutations_path)

omics_profiles = pd.read_csv(omics_profiles_path)

gdsc = pd.read_excel(gdsc_path)

print(f"Harmonized models : {harmonized_models.shape}")
print(f"Expression         : {expression.shape}")
print(f"Mutations          : {mutations.shape}")
print(f"Omics profiles     : {omics_profiles.shape}")
print(f"GDSC               : {gdsc.shape}")

C:\Users\paula\AppData\Local\Temp\ipykernel_18260\2160714437.py:49: DtypeWarning: Columns (28,52,56,58,59,61) have mixed types. Specify dtype option on import or set low_memory=False.
  mutations = pd.read_csv(mutations_path)


Harmonized models : (968, 8)
Expression         : (1754, 19221)
Mutations          : (1177454, 69)
Omics profiles     : (4775, 18)
GDSC               : (242036, 19)


In [3]:
datasets = {
    "harmonized_models": harmonized_models,
    "expression": expression,
    "mutations": mutations,
    "omics_profiles": omics_profiles,
    "gdsc": gdsc,
}

for name, df in datasets.items():
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print(df.head(3))


harmonized_models
      ModelID SangerModelID  COSMICID CellLineName       OncotreeLineage  \
0  ACH-000001     SIDM00105  905933.0  NIH:OVCAR-3  Ovary/Fallopian Tube   
1  ACH-000002     SIDM00829  905938.0        HL-60               Myeloid   
2  ACH-000004     SIDM00594  907053.0          HEL               Myeloid   

     OncotreePrimaryDisease                          OncotreeSubtype  \
0  Ovarian Epithelial Tumor         High-Grade Serous Ovarian Cancer   
1    Acute Myeloid Leukemia  AML with Myelodysplasia-Related Changes   
2    Acute Myeloid Leukemia                                 AML, NOS   

                                  CCLEName  
0                          NIHOVCAR3_OVARY  
1  HL60_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE  
2   HEL_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE  

expression
   Unnamed: 0 SequencingID     ModelID IsDefaultEntryForModel  \
0           0   CDS-010xbm  ACH-001113                    Yes   
1           1   CDS-02TzJp  ACH-001289                    Yes   


In [5]:
# =============================================================================
# Column inspection
# =============================================================================

for name, df in {
    "harmonized_models": harmonized_models,
    "expression": expression,
    "mutations": mutations,
    "omics_profiles": omics_profiles,
    "gdsc": gdsc,
}.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    print(list(df.columns[:100]))


harmonized_models
['ModelID', 'SangerModelID', 'COSMICID', 'CellLineName', 'OncotreeLineage', 'OncotreePrimaryDisease', 'OncotreeSubtype', 'CCLEName']

expression
['Unnamed: 0', 'SequencingID', 'ModelID', 'IsDefaultEntryForModel', 'ModelConditionID', 'IsDefaultEntryForMC', 'TSPAN6 (7105)', 'TNMD (64102)', 'DPM1 (8813)', 'SCYL3 (57147)', 'FIRRM (55732)', 'FGR (2268)', 'CFH (3075)', 'FUCA2 (2519)', 'GCLC (2729)', 'NFYA (4800)', 'STPG1 (90529)', 'NIPAL3 (57185)', 'LAS1L (81887)', 'ENPP4 (22875)', 'SEMA3F (6405)', 'CFTR (1080)', 'ANKIB1 (54467)', 'CYP51A1 (1595)', 'KRIT1 (889)', 'RAD52 (5893)', 'MYH16 (84176)', 'BAD (572)', 'LAP3 (51056)', 'CD99 (4267)', 'HS3ST1 (9957)', 'AOC1 (26)', 'WNT16 (51384)', 'HECW1 (23072)', 'MAD1L1 (8379)', 'LASP1 (3927)', 'SNX11 (29916)', 'TMEM176A (55365)', 'M6PR (4074)', 'KLHL13 (90293)', 'CYP26B1 (56603)', 'ICA1 (3382)', 'DBNDD1 (79007)', 'ALS2 (57679)', 'CASP10 (843)', 'CFLAR (8837)', 'TFPI (7035)', 'NDUFAF7 (55471)', 'RBM5 (10181)', 'MTMR7 (9108)', 'SLC7A2

In [6]:
omics_profiles["DataType"].value_counts()

DataType
wes    1912
rna    1754
wgs    1109
Name: count, dtype: int64

In [7]:
# Coverage audit
expr_models = set(expression["ModelID"])

mut_models = set(mutations["ModelID"])

omics_models = set(omics_profiles["ModelID"])

gdsc_models = set(
    harmonized_models.loc[
        harmonized_models["SangerModelID"].isin(
            gdsc["SANGER_MODEL_ID"]
        ),
        "ModelID"
    ]
)

In [9]:
omics_profiles.groupby("DataType")["ModelID"].nunique()

DataType
rna    1699
wes    1703
wgs    1095
Name: ModelID, dtype: int64

In [10]:
# Summary of coverage
omics_availability = (
    pd.crosstab(
        omics_profiles["ModelID"],
        omics_profiles["DataType"]
    )
    .gt(0)
    .astype(int)
)

omics_availability.head()

DataType,rna,wes,wgs
ModelID,,,
ACH-000001,1,1,1
ACH-000002,1,1,0
ACH-000003,1,0,0
ACH-000004,1,1,1
ACH-000005,1,1,1


In [11]:
omics_availability.sum()

DataType
rna    1699
wes    1703
wgs    1095
dtype: int64

In [12]:
omics_availability.mean() * 100

DataType
rna    83.653373
wes    83.850320
wgs    53.914328
dtype: float64

## Coverage within the harmonized model universe

Assess the availability of molecular and pharmacological data layers
across the harmonized DepMap–GDSC model universe established in
Notebook 21.

Coverage is evaluated relative to the 968 harmonized models and serves
as the basis for defining the final integrated modeling cohort.

In [13]:
# =============================================================================
# Harmonized universe
# =============================================================================

harmonized_model_ids = set(
    harmonized_models["ModelID"]
)

# =============================================================================
# Available models by data layer
# =============================================================================

expr_models = set(
    expression["ModelID"].dropna().unique()
)

mut_models = set(
    mutations["ModelID"].dropna().unique()
)

omics_models = set(
    omics_profiles["ModelID"].dropna().unique()
)

gdsc_sanger_ids = set(
    gdsc["SANGER_MODEL_ID"].dropna().unique()
)

gdsc_models = set(
    harmonized_models.loc[
        harmonized_models["SangerModelID"].isin(gdsc_sanger_ids),
        "ModelID"
    ]
)

# =============================================================================
# Coverage summary
# =============================================================================

coverage_summary = pd.DataFrame({
    "Layer": [
        "Expression",
        "Mutations",
        "Omics Profiles",
        "Drug Response"
    ],
    "Models": [
        len(harmonized_model_ids & expr_models),
        len(harmonized_model_ids & mut_models),
        len(harmonized_model_ids & omics_models),
        len(harmonized_model_ids & gdsc_models),
    ]
})

coverage_summary["CoveragePct"] = (
    coverage_summary["Models"]
    / len(harmonized_model_ids)
    * 100
).round(2)

coverage_summary

,Layer,Models,CoveragePct
0,Expression,715,73.86
1,Mutations,959,99.07
2,Omics Profiles,960,99.17
3,Drug Response,968,100.00


In [14]:
missing_expression = (
    harmonized_model_ids
    - expr_models
)

len(missing_expression)

253

In [18]:
print(harmonized_models.loc[
    harmonized_models["ModelID"].isin(
        list(missing_expression)
    )
].head(20))

        ModelID SangerModelID   COSMICID CellLineName       OncotreeLineage  \
21   ACH-000047     SIDM00274   906869.0         GCIY     Esophagus/Stomach   
177  ACH-000309     SIDM01108   909721.0      SK-LU-1                  Lung   
314  ACH-000534     SIDM00413  1331050.0    WSU-DLCL2              Lymphoid   
440  ACH-000727     SIDM00711  1330973.0    NCI-H2066                  Lung   
482  ACH-000795     SIDM00433   908146.0      MOLT-13              Lymphoid   
623  ACH-001002     SIDM01240  1287706.0        451Lu                  Skin   
625  ACH-001021     SIDM01237   910850.0          C3A                 Liver   
626  ACH-001023     SIDM01248   910568.0     CGTH-W-1         Head and Neck   
627  ACH-001024     SIDM01228   910853.0        CHL-1                  Skin   
628  ACH-001039     SIDM00826   905961.0     COLO 205                 Bowel   
629  ACH-001049     SIDM00953   753547.0        CPC-N                  Lung   
630  ACH-001063     SIDM00969  1479987.0        DOV1

In [16]:
expression["IsDefaultEntryForModel"].value_counts()

IsDefaultEntryForModel
Yes    1699
No       55
Name: count, dtype: int64

In [17]:
expression_default = expression[
    expression["IsDefaultEntryForModel"] == "Yes"
]

expression_default["ModelID"].nunique()

1699

In [19]:
expression["ModelID"].head()

0    ACH-001113
1    ACH-001289
2    ACH-001339
3    ACH-001619
4    ACH-001979
Name: ModelID, dtype: object

In [20]:
expression["ModelID"].tail()

1749    ACH-002669
1750    ACH-001858
1751    ACH-001997
1752    ACH-003297
1753    ACH-000052
Name: ModelID, dtype: object

In [21]:
missing_expression_df = harmonized_models[
    harmonized_models["ModelID"].isin(missing_expression)
]

In [22]:
missing_expression_df["OncotreeLineage"].value_counts()

OncotreeLineage
Lymphoid                     40
Lung                         32
Bone                         22
CNS/Brain                    18
Skin                         17
Head and Neck                16
Kidney                       16
Pleura                       14
Peripheral Nervous System    13
Esophagus/Stomach            11
Myeloid                      11
Ovary/Fallopian Tube          7
Bowel                         6
Soft Tissue                   5
Breast                        5
Cervix                        3
Pancreas                      3
Vulva/Vagina                  3
Liver                         2
Biliary Tract                 2
Thyroid                       2
Bladder/Urinary Tract         2
Prostate                      1
Other                         1
Testis                        1
Name: count, dtype: int64

In [23]:
missing_expression_df["OncotreePrimaryDisease"].value_counts().head(20)

OncotreePrimaryDisease
Lung Neuroendocrine Tumor                18
Diffuse Glioma                           18
Mature B-Cell Neoplasms                  17
Ewing Sarcoma                            17
Melanoma                                 16
Renal Cell Carcinoma                     16
Pleural Mesothelioma                     14
Non-Small Cell Lung Cancer               14
Head and Neck Squamous Cell Carcinoma    14
Neuroblastoma                            13
T-Lymphoblastic Leukemia/Lymphoma         8
Esophagogastric Adenocarcinoma            7
B-Cell Acute Lymphoblastic Leukemia       7
Ovarian Epithelial Tumor                  6
Colorectal Adenocarcinoma                 6
Acute Myeloid Leukemia                    6
Invasive Breast Carcinoma                 5
Non-Hodgkin Lymphoma                      5
Esophageal Squamous Cell Carcinoma        4
Chondrosarcoma                            4
Name: count, dtype: int64

In [24]:
coverage_df = harmonized_models.copy()

coverage_df["has_expression"] = (
    coverage_df["ModelID"].isin(expr_models)
)

coverage_df["has_mutations"] = (
    coverage_df["ModelID"].isin(mut_models)
)

coverage_df["has_omics_profile"] = (
    coverage_df["ModelID"].isin(omics_models)
)

In [25]:
(
    coverage_df
    .groupby("OncotreeLineage")["has_expression"]
    .mean()
    .sort_values()
    * 100
)

OncotreeLineage
Other                          0.000000
Pleura                        33.333333
Vulva/Vagina                  40.000000
Bone                          46.341463
Biliary Tract                 50.000000
Kidney                        55.555556
Peripheral Nervous System     58.064516
Head and Neck                 61.904762
Soft Tissue                   66.666667
Testis                        66.666667
CNS/Brain                     67.272727
Lymphoid                      68.253968
Skin                          68.518519
Myeloid                       73.809524
Cervix                        80.000000
Lung                          80.952381
Esophagus/Stomach             82.258065
Ovary/Fallopian Tube          82.926829
Thyroid                       86.666667
Liver                         86.666667
Prostate                      87.500000
Bowel                         87.755102
Bladder/Urinary Tract         88.888889
Pancreas                      90.322581
Breast                  

## RNA-seq coverage loss within the harmonized universe

Expression availability is the main source of model loss within the harmonized DepMap–GDSC universe.

Although mutation data, omics profile metadata, and GDSC pharmacological response show near-complete coverage, only 715 of 968 harmonized models have expression data available in the current DepMap expression matrix.

This loss does not appear to be caused by malformed ModelID values or by non-default expression profiles. Instead, it reflects data-layer availability differences across resources.

Because downstream transcriptomic and pharmacogenomic analyses require expression data, this coverage loss must be explicitly documented before defining the integrated modeling cohort.